In [15]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# 1. Define Application State
class TransferState(TypedDict):
    amount: float
    recipient: str
    approved: bool
    status: str

# 2. Define Nodes
def prepare_transfer(state: TransferState):
    print(f"\n[Node: prepare] Processing transfer request...")
    print(f" Amount: ${state['amount']} |  Recipient: {state['recipient']}")
    return {"status": "pending_review"}

def execute_transfer(state: TransferState):
    print(f"\n[Node: execute] Finalizing transaction...")
    if state.get("approved"):
        print(" [SUCCESS] Money transferred successfully! Transaction complete.")
        return {"status": "completed"}
    else:
        print(" [BLOCKED] Unauthorized or Rejected transfer attempt!")
        return {"status": "failed"}
# 3. Assemble and Compile the Graph
builder = StateGraph(TransferState)
builder.add_node("prepare", prepare_transfer)
builder.add_node("execute", execute_transfer)

builder.add_edge(START, "prepare")
builder.add_edge("prepare", "execute")
builder.add_edge("execute", END)

# Initialize memory for checkpointing
memory = MemorySaver()

# Crucial: Automatically pause BEFORE the 'execute' node runs
app = builder.compile(checkpointer=memory, interrupt_before=["execute"])

In [16]:
#  Initial Execution (Triggers the Interrupt)
config = {"configurable": {"thread_id": "nb_transfer_101"}}
initial_input = {"amount": 12550.00, "recipient": "Global Tech Corp", "approved": False, "status": "initiated"}

print(" Initiating the transfer workflow...")
app.invoke(initial_input, config)

print("\n WORKFLOW INTERRUPTED: The agent is paused and waiting for human sign-off.")

 Initiating the transfer workflow...

[Node: prepare] Processing transfer request...
 Amount: $12550.0 |  Recipient: Global Tech Corp

 WORKFLOW INTERRUPTED: The agent is paused and waiting for human sign-off.


In [1]:
# Interactive Human Review and Resume
# Get current state from the checkpointer
current_state = app.get_state(config)

if current_state.next: # Checks if the graph is currently paused at a node
    print(" --- PENDING TRANSACTION REVIEW ---")
    print(f"Vendor: {current_state.values['recipient']}")
    print(f"Amount: ${current_state.values['amount']}")
    print("-------------------------------------")
    
    # Jupyter notebook's text input for interaction
    user_choice = input("Type 'Y' to Approve, 'N' to Reject: ").strip().upper()
    
    if user_choice == 'Y':
        # Update the state with approved=True
        app.update_state(config, {"approved": True, "status": "human_approved"}, as_node="prepare")
        print("\n Transfer Approved! Resuming graph execution...")
        app.invoke(None, config) # Passing None tells LangGraph to pick up right where it left off
    else:
        # Update the state with approved=False
        app.update_state(config, {"approved": False, "status": "human_rejected"}, as_node="prepare")
        print("\n Transfer Rejected! Resuming graph to safely cancel...")
        app.invoke(None, config)
else:
    print("No pending transactions awaiting approval.")

NameError: name 'app' is not defined